# 02 - Train and Infer GeoRT Student

This notebook calls the training and inference scripts. Model, losses, metrics, and IO live in `src/`.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/GeoRT')
except Exception:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
%pip install -r requirements.txt

In [ ]:
from src.utils import load_project_config
from src.dataset import KITTIDepthCompletionDataset
from src.prepare_depth_selection import build_depth_selection_splits

cfg, paths = load_project_config('configs/geort_student_s.yaml')
build_depth_selection_splits(paths['data_root'], train_count=800)
train_ds = KITTIDepthCompletionDataset(
    paths['data_root'], paths['split_root'], paths['train_split'], 'train',
    image_size=tuple(cfg['data']['image_size']), output_scale=cfg['data']['output_scale'],
    teacher_root=paths['teacher_root'], load_teacher=True, return_tensors=False,
)
print('train samples:', len(train_ds))
sample = train_ds.load_sample_np(0)
print(sample['sample_id'], sample['rgb'].shape, sample['sparse'].shape)

In [ ]:
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from src.sparse_propagation import fast_sparse_propagation
from src.utils import load_npz_array

sample = train_ds.load_sample_np(0)
rgb = torch.from_numpy(sample['rgb'].transpose(2,0,1)).float()[None] / 255.0
sparse = torch.from_numpy(sample['sparse'][None,None]).float()
mask = torch.from_numpy(sample['mask'][None,None]).float()
ray = torch.from_numpy(sample['ray'][None]).float()
D_init = fast_sparse_propagation(rgb, sparse, mask, ray, scale=4, k=4)[0,0].numpy()

sid = sample['sample_id']
fused_path = Path(paths['teacher_root']) / 'fused' / 'train' / f'{sid}.npz'
D_teacher = load_npz_array(fused_path, 'D_teacher') if fused_path.exists() else D_init * 0
C_teacher = load_npz_array(fused_path, 'C_teacher') if fused_path.exists() else D_init * 0

fig, ax = plt.subplots(1, 5, figsize=(20, 4))
ax[0].imshow(sample['rgb']); ax[0].set_title('RGB')
ax[1].imshow(sample['sparse'], cmap='magma'); ax[1].set_title('Sparse')
ax[2].imshow(D_init, cmap='magma'); ax[2].set_title('D_init')
ax[3].imshow(D_teacher, cmap='magma'); ax[3].set_title('D_teacher')
ax[4].imshow(C_teacher, cmap='viridis', vmin=0, vmax=1); ax[4].set_title('C_teacher')
for a in ax: a.axis('off')
plt.tight_layout()

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'src.train_student', '--config', 'configs/geort_student_s.yaml'], check=True)

In [ ]:
from pathlib import Path
best_ckpt = Path(paths['student_root']) / 'checkpoints' / 'best.pth'
print('best checkpoint:', best_ckpt, best_ckpt.exists())

In [ ]:
import subprocess, sys
best_ckpt = str(Path(paths['student_root']) / 'checkpoints' / 'best.pth')
subprocess.run([sys.executable, '-m', 'src.infer_student', '--config', 'configs/geort_student_s.yaml', '--checkpoint', best_ckpt, '--split', 'val'], check=True)
subprocess.run([sys.executable, '-m', 'src.infer_student', '--config', 'configs/geort_student_s.yaml', '--checkpoint', best_ckpt, '--split', 'test'], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

pred_dir = Path(paths['student_root']) / 'val_predictions'
pred_file = sorted(pred_dir.glob('*.npz'))[0]
with np.load(pred_file) as data:
    D_c = data['D_c']
    C = data['C']
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(D_c, cmap='magma'); ax[0].set_title('D_c')
ax[1].imshow(C, cmap='viridis'); ax[1].set_title('C')
for a in ax: a.axis('off')
plt.tight_layout()